# E-Commerce Advanced Business Cases

In diesem Notebook implementieren wir vier hochgradig praxisrelevante Data-Science & Business-Konzepte auf unserem Datensatz.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')

def lade_daten():
    return pd.read_csv("online_shoppers_intention.csv")

print("Bibliotheken und Helfer geladen!")

## 1. Smart Vouchers (`predict_proba`)
Wir verteilen Gutscheine nicht blind an alle, sondern **nur an Kunden, deren Kaufwahrscheinlichkeit zwischen 40% und 60% liegt** (die Unentschlossenen).

In [ ]:
df = lade_daten()
df_encoded = pd.get_dummies(df, drop_first=True)
X = df_encoded.drop(columns=['Revenue'])
y = df_encoded['Revenue'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

mlp = MLPClassifier(hidden_layer_sizes=(16, 8), random_state=42, max_iter=300)
mlp.fit(X_train_scaled, y_train)

# WAHRSCHEINLICHKEITEN ABRUFEN (predict_proba gibt für jeden Nutzer [Prob_Nein, Prob_Ja] zurück)
kauf_wahrscheinlichkeiten = mlp.predict_proba(X_test_scaled)[:, 1] # Wir wollen nur die 'Ja'-Wahrscheinlichkeit

# Unentschlossene herausfiltern (40% - 60%)
unentschlossen_mask = (kauf_wahrscheinlichkeiten >= 0.40) & (kauf_wahrscheinlichkeiten <= 0.60)
kunden_fuer_gutschein = X_test[unentschlossen_mask]

print(f"Von {len(X_test)} Test-Besuchern sind genau {len(kunden_fuer_gutschein)} 'auf der Kippe'.")
print("Diesen Kunden zeigen wir sofort ein Rabatt-Popup an!")
print("Beispiel-Kunden (Index-Nummern):", kunden_fuer_gutschein.index[:5].tolist())

## 2. Feature Selection (Ballast abwerfen)
Muss die KI wirklich über 60 Spalten (nach dem Encoding) durchkauen? Wir testen, ob ein winziges Modell mit nur den **Top 3 Features** genauso gut ist.

In [ ]:
# Nur die Top 3 Spalten nutzen
top_features = ['PageValues', 'ExitRates', 'ProductRelated_Duration']
X_top = df[top_features]

X_train_top, X_test_top, _, _ = train_test_split(X_top, y, test_size=0.2, random_state=42)
scaler_top = StandardScaler()
X_train_top_scaled = scaler_top.fit_transform(X_train_top)
X_test_top_scaled = scaler_top.transform(X_test_top)

mlp_top = MLPClassifier(hidden_layer_sizes=(16, 8), random_state=42, max_iter=300)
mlp_top.fit(X_train_top_scaled, y_train)

acc_full = accuracy_score(y_test, mlp.predict(X_test_scaled))
acc_top = accuracy_score(y_test, mlp_top.predict(X_test_top_scaled))

print(f"Genauigkeit mit ALLEN (über 60) Features: {acc_full*100:.1f}%")
print(f"Genauigkeit mit NUR 3 Features: {acc_top*100:.1f}%")
print("=> Erkenntnis: Wir erreichen fast die gleiche Leistung mit einem Bruchteil der Daten. Das spart Server-Kosten!")

## 3. Saisonale Modelle (Trainingsdaten aufteilen)
Das Kundenverhalten am Black Friday (November) ist komplett anders als im Februar. Wir splitten die Daten.

In [ ]:
df = lade_daten()

# In November/Dezember (Q4) und den Rest aufteilen
df_q4 = df[df['Month'].isin(['Nov', 'Dec'])]
df_rest = df[~df['Month'].isin(['Nov', 'Dec'])]

print(f"Besucher im umsatzstarken Q4 (Nov/Dec): {len(df_q4)}")
print(f"Besucher im restlichen Jahr: {len(df_rest)}")
print("\nIn der echten Welt würden wir nun zwei völlig getrennte KNNs für diese Zeiträume trainieren, da die Kaufmuster unterschiedlich sind.")

## 4. Anomalie-Erkennung (Bot-Detection)
Nicht jeder Besucher ist ein Mensch. Viele sind Bots (Scraper). Mit einem `IsolationForest` suchen wir die 1% ungewöhnlichsten Besucher heraus.

In [ ]:
df = lade_daten()
numerische_spalten = df.select_dtypes(include=[np.number])

# Isolation Forest: Sucht die 1% (contamination=0.01) stärksten Ausreißer im Datensatz
iso_forest = IsolationForest(contamination=0.01, random_state=42)
anomalien = iso_forest.fit_predict(numerische_spalten)

# -1 bedeutet Anomalie (Bot), 1 bedeutet normaler Mensch
bots = df[anomalien == -1]

print(f"Der Algorithmus hat {len(bots)} extrem ungewöhnliche Besucher gefunden.")
print("Auffällig: Diese User rufen oft hunderte Seiten auf ('ProductRelated'), aber die Verweildauer ('Duration') passt nicht dazu.")
display(bots[['ProductRelated', 'ProductRelated_Duration', 'PageValues']].head(5))